# OPTIMIZED PIPELINE 3: FINANCIAL DATA EXTRACTION

## Strategy
1. Replicate the EXACT same row-filtering pipeline as Notebook 01 (dropping ALL 12 leakage columns)
2. Save the surviving row indices
3. Go back to the raw CSV and extract recovery columns ONLY for those rows
4. Apply the same train_test_split to guarantee perfect index alignment

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

DATA_DIR = r"c:\Users\edgib\Downloads\PREDICTIVE MODEL USING NEURAL NETWORK\PHASE II"
INPUT_FILE = os.path.join(DATA_DIR, "ORIGINAL_DATASET_PHASE_II_KAGGLE.csv")
OUT_DIR = r"c:\Users\edgib\Downloads\PREDICTIVE MODEL USING NEURAL NETWORK\OPTIMIZED_CLEANING_PIPELINES"
RANDOM_SEED = 42
COST_OF_FUNDS_RATE = 0.03

In [4]:
# 1. LOAD RAW DATA
df = pd.read_csv(INPUT_FILE, low_memory=False)

## PHASE A: Replicate the exact row-filtering from Pipeline 01
The goal is to end up with the exact same set of rows (same pandas index) as Notebook 01, so that train_test_split with the same random_state produces identical train/test partitions.

In [5]:
# 2. FILTER RESOLVED LOANS & CREATE TARGET
resolved_statuses = ["Fully Paid", "Charged Off"]
df = df[df["loan_status"].isin(resolved_statuses)].copy()
df["target"] = (df["loan_status"] == "Charged Off").astype(int)
df.drop(columns=["loan_status"], inplace=True)

In [6]:
# 3. DROP COLUMNS WITH 100% MISSING
df.drop(columns=df.columns[df.isnull().mean() == 1], inplace=True)

In [7]:
# 4. DROP CONSTANT COLUMNS
df.drop(columns=[col for col in df.columns if df[col].nunique() == 1], inplace=True)

In [8]:
# 5. DROP DATA-LEAKAGE COLUMNS (ALL 12 - exactly as Pipeline 01)
hardship_cols = [col for col in df.columns if "hardship" in col.lower() or "settlement" in col.lower()]
df.drop(columns=hardship_cols, inplace=True)

time_after_loan_cols = ["total_pymnt","total_pymnt_inv","total_rec_prncp","total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low"]
df.drop(columns=time_after_loan_cols, inplace=True, errors="ignore")

immediately_after_approval_columns = ["funded_amnt", "funded_amnt_inv", "url", "initial_list_status", "disbursement_method"]
df.drop(columns=immediately_after_approval_columns, inplace=True, errors="ignore")

In [9]:
# 6. DROP ID AND FREE-TEXT COLUMNS
df.drop(columns=["id","emp_title","title","zip_code","desc"], inplace=True, errors="ignore")

In [11]:
# 7. DROP COLUMNS WITH >40% MISSING
# Getting rid of all columns with less than 60% of available data:

# First we need to calculate the missing percentage for each column:
missing_percentages = (df.isnull().sum()/len(df) * 100) 
# df.isnull().sum() counts how many empty NaN rows exist in each column.
# /len(df) divides that count by the total number of rows in your entire dataset.

high_missing = missing_percentages[missing_percentages > 40]
# This filters the sorted list keeping only the columns with more than 40% data missing.
# now we put the lables of those columns into a list:
high_missing_columns = list(high_missing.index)

df.drop(columns = high_missing_columns,inplace = True)

In [13]:
# 8. CATEGORICAL ENCODING (only what affects row filtering)
df["term"] = df["term"].astype(str).str.strip().str.replace("months","").astype(int)
df.dropna(subset=["emp_length"], inplace=True)

In [14]:
# 9. DROP ROWS WITH <1% NULL COLUMNS
cols_less_1 = df.columns[(df.isnull().mean() > 0) & (df.isnull().mean() <= 0.01)]
df.dropna(subset=cols_less_1, inplace=True)

In [15]:
# 10. DROP ROWS WITH >6 NULLS
df = df[df.isnull().sum(axis=1) <= 6].copy()

## PHASE B: Capture surviving indices and recover financial columns
At this point, `df.index` contains the exact same row indices as Pipeline 01. We now go back to the raw CSV to fetch the recovery columns for these specific rows.

In [16]:
# 11. SAVE THE SURVIVING INDICES
surviving_indices = df.index
print(f"Surviving rows: {len(surviving_indices)}")

Surviving rows: 1200551


In [17]:
# 12. RELOAD RAW DATA AND EXTRACT FINANCIAL COLUMNS FOR SURVIVING ROWS ONLY
df_raw = pd.read_csv(INPUT_FILE, low_memory=False,
    usecols=["loan_amnt", "int_rate", "term",
             "total_pymnt", "total_rec_prncp", "total_rec_int",
             "total_rec_late_fee", "recoveries", "collection_recovery_fee"])

# Keep only the rows that survived filtering
df_finance = df_raw.loc[surviving_indices].copy()

# Clean term (same as pipeline)
df_finance["term"] = df_finance["term"].astype(str).str.strip().str.replace("months","").astype(int)

print(f"Financial rows extracted: {len(df_finance)}")

Financial rows extracted: 1200551


In [18]:
# 13. COMPUTE FINANCIAL METRICS
finance_cols = df_finance[["loan_amnt", "int_rate", "term"]].copy()

# A. EXPECTED PROFIT (French Amortization)
r = (df_finance["int_rate"] / 100.0) / 12.0
n = df_finance["term"]
pmt = df_finance["loan_amnt"] * (r * (1 + r)**n) / ((1 + r)**n - 1)
expected_interest = (pmt * n) - df_finance["loan_amnt"]

c = COST_OF_FUNDS_RATE / 12.0
cof_pmt = df_finance["loan_amnt"] * (c * (1 + c)**n) / ((1 + c)**n - 1)
expected_cof = (cof_pmt * n) - df_finance["loan_amnt"]

finance_cols["expected_profit"] = expected_interest - expected_cof

# B. REAL LOSS GIVEN DEFAULT
cash_in = (df_finance["total_rec_prncp"] + df_finance["total_rec_int"]
           + df_finance["total_rec_late_fee"]
           + (df_finance["recoveries"] - df_finance["collection_recovery_fee"]))
months_active = np.clip(df_finance["total_pymnt"] / pmt, 0, n)
cof_paid = expected_cof * (months_active / n)
cash_out = df_finance["loan_amnt"] + cof_paid

finance_cols["real_lgd"] = np.clip(cash_out - cash_in, 0, None)

In [19]:
# 14. TRAIN / TEST SPLIT (SAME SEED + SAME INDICES = GUARANTEED ALIGNMENT)
y = df["target"]  # from the filtered df (same rows, same order)

finance_train, finance_test, _, _ = train_test_split(
    finance_cols, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)

print(f"FINANCE_TRAIN: {finance_train.shape}")
print(f"FINANCE_TEST:  {finance_test.shape}")

FINANCE_TRAIN: (960440, 5)
FINANCE_TEST:  (240111, 5)


In [21]:
# 15. EXPORT
finance_train.to_csv(os.path.join(OUT_DIR, "FINANCE_TRAIN.csv"), index=False)
finance_test.to_csv(os.path.join(OUT_DIR, "FINANCE_TEST.csv"), index=False)

finance_train.to_parquet(os.path.join(OUT_DIR, "FINANCE_TRAIN.parquet"), index=False)
finance_test.to_parquet(os.path.join(OUT_DIR, "FINANCE_TEST.parquet"), index=False)
print("Financial Side-Channel Exported.")

Financial Side-Channel Exported.
